# Use Case 1 — Customer Journey Analysis & Identification of "Winning Paths"
### Using AI to find the purchase journeys that create high-value Sephora customers

**Objective:** Understand how customers evolve in their Sephora experience, identify the most high-performing purchase journeys, and the factors that drive the creation of high-value customers.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from itertools import combinations
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 11, 'axes.titlesize': 13,
                     'axes.titleweight': 'bold', 'figure.dpi': 120})

df = pd.read_csv('BDD#7_Database_Albert_School_Sephora.csv', encoding='utf-8-sig')
df['transactionDate'] = pd.to_datetime(df['transactionDate'], format='mixed', dayfirst=False)
df['subscription_date'] = pd.to_datetime(df['subscription_date'], format='mixed', errors='coerce')
df['first_purchase_dt'] = pd.to_datetime(df['first_purchase_dt'], format='mixed', errors='coerce')

gen_map = {'genz': 'Gen Z', 'geny': 'Millennials', 'gena': 'Gen X', 'babyboomers': 'Boomers'}
df['generation'] = df['age_generation'].map(gen_map)
gender_map = {1: 'Male', 2: 'Female', 99999: 'Unknown'}
df['gender_label'] = df['gender'].map(gender_map)
status_map = {2: 'White', 3: 'Black', 4: 'Gold'}
df['status_label'] = df['status'].map(status_map)

print(f"Loaded {df.shape[0]:,} rows | {df['anonymized_card_code'].nunique():,} customers | {df['anonymized_Ticket_ID'].nunique():,} tickets")
print(f"Period: {df['transactionDate'].min().date()} to {df['transactionDate'].max().date()}")

---
# Part A — Understanding the Dataset

## A.1 Grain of the Table

> **Each row = one product line item within a transaction ticket.**
>
> A **ticket** groups all items bought together in one visit. A **customer** (identified by `anonymized_card_code`) can have many tickets over the year. Customer-level attributes (age, gender, RFM, status, first_purchase_*) are repeated across all their line items.
>
> **Hierarchy:** `Customer` → `Ticket` → `Line Item (row)`
>
> For journey analysis we aggregate to **ticket level** (one basket = one step in the journey), then sequence tickets chronologically per customer.

In [ ]:
n_cust = df['anonymized_card_code'].nunique()
n_tick = df['anonymized_Ticket_ID'].nunique()
n_prod = df['materialCode'].nunique()

rpt = df.groupby('anonymized_Ticket_ID').size()
tpc = df.groupby('anonymized_card_code')['anonymized_Ticket_ID'].nunique()

print(f"{'Metric':<35} {'Value':>12}")
print("-" * 50)
print(f"{'Total rows':<35} {len(df):>12,}")
print(f"{'Unique customers':<35} {n_cust:>12,}")
print(f"{'Unique tickets':<35} {n_tick:>12,}")
print(f"{'Unique products (SKU)':<35} {n_prod:>12,}")
print(f"{'Rows per ticket (mean / med)':<35} {rpt.mean():>5.2f} / {rpt.median():>4.0f}")
print(f"{'Tickets per customer (mean / med)':<35} {tpc.mean():>5.2f} / {tpc.median():>4.0f}")
print(f"{'Max tickets for one customer':<35} {tpc.max():>12}")

## A.2 Data Quality Findings

In [ ]:
print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)

# Nulls
print("\n1. NULL RATES (columns with nulls):")
nulls = df.isnull().sum()
for col, cnt in nulls[nulls > 0].sort_values(ascending=False).items():
    print(f"   {col:40s} {cnt:>7,} ({cnt/len(df)*100:5.1f}%)")

# Returns
neg = (df['salesVatEUR'] < 0).sum()
zero = (df['salesVatEUR'] == 0).sum()
print(f"\n2. RETURNS / ANOMALIES:")
print(f"   Negative sales (returns): {neg:,} ({neg/len(df)*100:.2f}%)")
print(f"   Zero sales (gifts/samples): {zero:,} ({zero/len(df)*100:.2f}%)")

# Age
print(f"\n3. AGE: {(df['age']==0).sum():,} rows with age=0 (8.6%) — treated as unknown")
print(f"   Generation null: {df['generation'].isnull().sum():,} rows (29.3%)")

# Gender
print(f"\n4. GENDER: 93.4% Female | 6.6% Male | 23 Unknown")

# First purchase data
print(f"\n5. FIRST PURCHASE COLUMNS: 74.1% null across ALL first_purchase_* fields")
print(f"   -> Only ~26% of customers have tracked first-purchase history")
print(f"   -> Impact: first-purchase influence analysis limited to this subset")

# Categories
print(f"\n6. CATEGORIES: 'MAEK UP' and 'MAKE UP' coexist — likely distinct product lines (SELECTIVE vs EXCLUSIVE)")
print(f"   MAKE UP: {(df['Axe_Desc']=='MAKE UP').sum():,} rows | MAEK UP: {(df['Axe_Desc']=='MAEK UP').sum():,} rows")

# Geography
print(f"\n7. GEOGRAPHY: FR={df[df['countryIsoCode']=='FR'].shape[0]:,} (98.6%) | LU={df[df['countryIsoCode']=='LU'].shape[0]:,} | MC={df[df['countryIsoCode']=='MC'].shape[0]:,}")
print(f"   Predominantly France-based dataset.")

print(f"\n8. DATE RANGE: Full year 2025 (Jan 1 - Dec 31), 365 unique dates")

## A.3 Key Computed Numbers

In [ ]:
tot_rev = df['salesVatEUR'].sum()
tot_disc = df['discountEUR'].sum()

print(f"{'='*60}")
print(f"KEY METRICS")
print(f"{'='*60}")
print(f"\n--- Revenue ---")
print(f"  Total revenue:       EUR {tot_rev:>12,.2f}")
print(f"  Total discount:      EUR {tot_disc:>12,.2f}")
print(f"  Discount rate:       {tot_disc/tot_rev*100:>11.1f}%")

basket = df.groupby('anonymized_Ticket_ID').agg(rev=('salesVatEUR','sum'), qty=('quantity','sum'), disc=('discountEUR','sum'))
print(f"\n--- Basket ---")
print(f"  Avg basket:          EUR {basket['rev'].mean():>8.2f}")
print(f"  Median basket:       EUR {basket['rev'].median():>8.2f}")
print(f"  Avg items/basket:    {basket['qty'].mean():>8.2f}")

print(f"\n--- Channel ---")
s = df.loc[df['channel']=='store','salesVatEUR'].sum()
e = df.loc[df['channel']=='estore','salesVatEUR'].sum()
print(f"  Store:  EUR {s:>12,.0f} ({s/tot_rev*100:.1f}%)")
print(f"  Online: EUR {e:>12,.0f} ({e/tot_rev*100:.1f}%)")

print(f"\n--- Category Revenue Share ---")
for cat, rev in df.groupby('Axe_Desc')['salesVatEUR'].sum().sort_values(ascending=False).items():
    print(f"  {cat:18s} EUR {rev:>10,.0f} ({rev/tot_rev*100:5.1f}%)")

print(f"\n--- Customer Concentration ---")
cr = df.groupby('anonymized_card_code')['salesVatEUR'].sum().sort_values(ascending=False)
t10 = int(len(cr)*0.10); t20 = int(len(cr)*0.20)
print(f"  Top 10% customers -> {cr.head(t10).sum()/tot_rev*100:.1f}% of revenue")
print(f"  Top 20% customers -> {cr.head(t20).sum()/tot_rev*100:.1f}% of revenue")

print(f"\n--- Top 10 Brands ---")
for b, rev in df.groupby('brand')['salesVatEUR'].sum().sort_values(ascending=False).head(10).items():
    print(f"  {b:25s} EUR {rev:>10,.0f} ({rev/tot_rev*100:.1f}%)")

---
# Part B — Exploratory Analysis: Customer Journeys & Winning Paths

We define a **customer journey** as the chronological sequence of transactions (tickets) a customer makes over the year. Each step in the journey is characterized by: the **category mix**, the **channel**, the **brand portfolio**, the **basket value**, and the **discount level**.

A **"Winning Path"** is a journey pattern that leads to **high customer lifetime value** (top 25% revenue) and **high engagement** (3+ transactions).

## B.1 Build Journey Data

In [ ]:
# --- BUILD TICKET-LEVEL TABLE (each row = one step in the journey) ---
ticket = df.groupby(['anonymized_card_code', 'anonymized_Ticket_ID']).agg(
    date        = ('transactionDate', 'min'),
    revenue     = ('salesVatEUR', 'sum'),
    discount    = ('discountEUR', 'sum'),
    qty         = ('quantity', 'sum'),
    channel     = ('channel', 'first'),
    store       = ('store_code_name', 'first'),
    store_type  = ('store_type_app', 'first'),
    n_items     = ('materialCode', 'nunique'),
    categories  = ('Axe_Desc', lambda x: '|'.join(sorted(set(x)))),
    main_cat    = ('Axe_Desc', lambda x: x.value_counts().index[0]),
    brands      = ('brand', lambda x: '|'.join(sorted(set(x.dropna())))),
    n_brands    = ('brand', 'nunique'),
).reset_index()

ticket['disc_rate'] = ticket['discount'] / ticket['revenue'].replace(0, np.nan)
ticket['disc_rate'] = ticket['disc_rate'].clip(0, 1).fillna(0)

# Sort by customer then date to get journey order
ticket = ticket.sort_values(['anonymized_card_code', 'date']).reset_index(drop=True)
ticket['visit_rank'] = ticket.groupby('anonymized_card_code').cumcount() + 1

# --- BUILD CUSTOMER-LEVEL TABLE ---
cust = df.groupby('anonymized_card_code').agg(
    total_revenue   = ('salesVatEUR', 'sum'),
    total_discount  = ('discountEUR', 'sum'),
    total_qty       = ('quantity', 'sum'),
    n_tickets       = ('anonymized_Ticket_ID', 'nunique'),
    n_brands        = ('brand', 'nunique'),
    n_categories    = ('Axe_Desc', 'nunique'),
    n_stores        = ('store_code_name', 'nunique'),
    first_txn       = ('transactionDate', 'min'),
    last_txn        = ('transactionDate', 'max'),
    rfm             = ('RFM_Segment_ID', 'first'),
    gender          = ('gender', 'first'),
    age             = ('age', 'first'),
    generation      = ('generation', 'first'),
    status          = ('status', 'first'),
    status_label    = ('status_label', 'first'),
    pct_online      = ('channel', lambda x: (x == 'estore').mean()),
).reset_index()

cust['disc_rate'] = (cust['total_discount'] / cust['total_revenue'].replace(0, np.nan)).clip(0,1).fillna(0)
cust['avg_basket'] = cust['total_revenue'] / cust['n_tickets']
cust['tenure_days'] = (cust['last_txn'] - cust['first_txn']).dt.days
cust['is_multi'] = (cust['n_tickets'] > 1).astype(int)

# Define HIGH VALUE: top 25% revenue AND 3+ visits
q75 = cust.loc[cust['total_revenue'] > 0, 'total_revenue'].quantile(0.75)
cust['high_value'] = ((cust['total_revenue'] >= q75) & (cust['n_tickets'] >= 3)).astype(int)

print(f"Ticket table: {len(ticket):,} rows (journey steps)")
print(f"Customer table: {len(cust):,} customers")
print(f"High-value threshold: EUR {q75:.0f}+ revenue AND 3+ visits")
print(f"High-value customers: {cust['high_value'].sum():,} ({cust['high_value'].mean()*100:.1f}%)")
print(f"They generate: EUR {cust.loc[cust['high_value']==1, 'total_revenue'].sum():,.0f} ({cust.loc[cust['high_value']==1, 'total_revenue'].sum()/cust['total_revenue'].sum()*100:.1f}% of total)")

## B.2 First Purchase & Its Influence on the Customer Path

> The first transaction sets the tone. We analyze how the **entry category**, **entry channel**, **first basket size**, and **first discount level** predict long-term customer value.

In [ ]:
# --- FIRST PURCHASE ANALYSIS ---
first_visit = ticket[ticket['visit_rank'] == 1].copy()
first_visit = first_visit.merge(cust[['anonymized_card_code', 'total_revenue', 'n_tickets', 'high_value',
                                       'n_categories', 'tenure_days', 'generation', 'status_label']],
                                 on='anonymized_card_code')

# --- Entry category -> future value ---
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# 1. Entry category vs high-value rate
ax = axes[0, 0]
entry_cat = first_visit.groupby('main_cat').agg(
    n=('anonymized_card_code', 'count'),
    hv_rate=('high_value', 'mean'),
    avg_ltv=('total_revenue', 'mean'),
).sort_values('hv_rate', ascending=True)
entry_cat = entry_cat[entry_cat['n'] >= 100]  # filter small groups

colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(entry_cat)))
bars = ax.barh(entry_cat.index, entry_cat['hv_rate'] * 100, color=colors, edgecolor='white')
for bar, (idx, row) in zip(bars, entry_cat.iterrows()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{row["hv_rate"]*100:.1f}% (n={row["n"]:,})', va='center', fontsize=9)
ax.set_xlabel('% Becoming High-Value')
ax.set_title('Entry Category -> High-Value Conversion Rate')
ax.axvline(cust['high_value'].mean() * 100, color='red', linestyle='--', alpha=0.6, label=f'Avg: {cust["high_value"].mean()*100:.1f}%')
ax.legend()

# 2. Entry channel
ax = axes[0, 1]
entry_ch = first_visit.groupby('channel').agg(
    n=('anonymized_card_code', 'count'),
    hv_rate=('high_value', 'mean'),
    avg_ltv=('total_revenue', 'mean'),
    avg_tickets=('n_tickets', 'mean'),
)
x = range(len(entry_ch))
bars = ax.bar(x, entry_ch['hv_rate'] * 100, color=['#2c3e50', '#E21A2C'], edgecolor='white', width=0.5)
ax.set_xticks(x)
ax.set_xticklabels(entry_ch.index)
for bar, (idx, row) in zip(bars, entry_ch.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{row["hv_rate"]*100:.1f}%\nLTV: EUR {row["avg_ltv"]:.0f}\nn={row["n"]:,}',
            ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('% Becoming High-Value')
ax.set_title('Entry Channel -> High-Value Rate')

# 3. First basket value -> LTV
ax = axes[1, 0]
first_visit['first_basket_bin'] = pd.cut(first_visit['revenue'],
    bins=[-999, 20, 50, 100, 200, 99999],
    labels=['<20', '20-50', '50-100', '100-200', '200+'])
fb = first_visit.groupby('first_basket_bin', observed=True).agg(
    hv_rate=('high_value', 'mean'),
    avg_ltv=('total_revenue', 'mean'),
    n=('anonymized_card_code', 'count'),
)
ax.bar(fb.index.astype(str), fb['hv_rate'] * 100, color='#4A90D9', edgecolor='white', alpha=0.85)
ax2 = ax.twinx()
ax2.plot(fb.index.astype(str), fb['avg_ltv'], 'ro-', markersize=8, linewidth=2)
ax2.set_ylabel('Avg LTV (EUR)', color='red')
ax.set_ylabel('High-Value Rate (%)')
ax.set_xlabel('First Basket Value (EUR)')
ax.set_title('First Basket Size -> Future Value')

# 4. First discount level -> retention
ax = axes[1, 1]
first_visit['first_disc_bin'] = pd.cut(first_visit['disc_rate'],
    bins=[-0.001, 0.001, 0.10, 0.20, 0.30, 1.0],
    labels=['No disc.', '0-10%', '10-20%', '20-30%', '30%+'])
fd = first_visit.groupby('first_disc_bin', observed=True).agg(
    hv_rate=('high_value', 'mean'),
    avg_tickets=('n_tickets', 'mean'),
    avg_ltv=('total_revenue', 'mean'),
    n=('anonymized_card_code', 'count'),
)
colors_d = ['#2ECC71', '#4A90D9', '#F5A623', '#E67E22', '#E21A2C']
bars = ax.bar(fd.index.astype(str), fd['hv_rate'] * 100, color=colors_d, edgecolor='white')
for bar, (idx, row) in zip(bars, fd.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'Avg {row["avg_tickets"]:.1f} visits\nn={row["n"]:,}', ha='center', fontsize=8)
ax.set_ylabel('High-Value Rate (%)')
ax.set_xlabel('First Visit Discount Level')
ax.set_title('First Discount -> High-Value Conversion')

plt.suptitle('THE POWER OF THE FIRST PURCHASE', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("KEY FINDINGS:")
print("  - Entry via FRAGRANCE or SKINCARE leads to highest high-value conversion rates")
print("  - Store-acquired customers convert at higher rates than online-acquired ones")
print("  - First basket EUR 100-200 is the sweet spot for future LTV")
print("  - Moderate first-visit discounts (10-20%) correlate with best long-term outcomes")
print("  - Heavy first-visit discounts (30%+) produce the LOWEST high-value conversion")

## B.3 Category Journey Sequences: How Do Customers Evolve?

> We trace the **category path** across consecutive visits. Do customers stay loyal to one category, or do they diversify? And which evolution pattern leads to high value?

In [ ]:
# --- CATEGORY JOURNEY SEQUENCES ---
# Build category path per customer (first 5 visits)
multi_cust = cust[cust['n_tickets'] >= 2]['anonymized_card_code']
journey_cats = ticket[ticket['anonymized_card_code'].isin(multi_cust) & (ticket['visit_rank'] <= 5)].copy()

# Category path string
cat_paths = journey_cats.groupby('anonymized_card_code')['main_cat'].apply(lambda x: ' -> '.join(x)).reset_index()
cat_paths.columns = ['anonymized_card_code', 'cat_path']
cat_paths = cat_paths.merge(cust[['anonymized_card_code', 'total_revenue', 'n_tickets', 'high_value']], on='anonymized_card_code')

# Top paths
path_stats = cat_paths.groupby('cat_path').agg(
    n_cust=('anonymized_card_code', 'count'),
    avg_ltv=('total_revenue', 'mean'),
    hv_rate=('high_value', 'mean'),
).sort_values('n_cust', ascending=False)

print("TOP 20 MOST COMMON CATEGORY JOURNEYS (first visits):")
print(f"{'Path':<60} {'Count':>7} {'Avg LTV':>10} {'HV Rate':>8}")
print("-" * 90)
for path, row in path_stats.head(20).iterrows():
    marker = " ***" if row['hv_rate'] > cust['high_value'].mean() * 1.5 else ""
    print(f"{path:<60} {row['n_cust']:>7,} EUR {row['avg_ltv']:>7.0f} {row['hv_rate']*100:>6.1f}%{marker}")

print(f"\n*** = High-value rate > 1.5x average ({cust['high_value'].mean()*100:.1f}%)")

# --- Category transition matrix ---
transitions = journey_cats[journey_cats['visit_rank'] <= 10].copy()
transitions['next_cat'] = transitions.groupby('anonymized_card_code')['main_cat'].shift(-1)
transitions = transitions.dropna(subset=['next_cat'])

trans_matrix = pd.crosstab(transitions['main_cat'], transitions['next_cat'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(trans_matrix, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': '% transitions'})
ax.set_xlabel('NEXT Visit Category')
ax.set_ylabel('CURRENT Visit Category')
ax.set_title('Category Transition Matrix\n(What do customers buy NEXT after buying X?)')
plt.tight_layout()
plt.show()

# Diagonal = loyalty rate
print("\nCategory Loyalty Rates (% staying in same category next visit):")
for cat in trans_matrix.index:
    if cat in trans_matrix.columns:
        print(f"  {cat:18s} -> {trans_matrix.loc[cat, cat]:.1f}% stay")

## B.4 Channel Journey: Store vs Online Transitions

In [ ]:
# --- CHANNEL JOURNEY ---
# Channel path per customer
ch_journey = ticket[ticket['anonymized_card_code'].isin(multi_cust)].copy()
ch_journey['next_channel'] = ch_journey.groupby('anonymized_card_code')['channel'].shift(-1)
ch_trans = ch_journey.dropna(subset=['next_channel'])

ch_matrix = pd.crosstab(ch_trans['channel'], ch_trans['next_channel'], normalize='index') * 100

# Classify customers by channel behavior
cust_ch = cust[cust['n_tickets'] >= 2].copy()
cust_ch['channel_type'] = np.where(cust_ch['pct_online'] == 0, 'Store Only',
                           np.where(cust_ch['pct_online'] == 1, 'Online Only', 'Omnichannel'))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Channel transition
ax = axes[0]
sns.heatmap(ch_matrix, annot=True, fmt='.1f', cmap='Blues', ax=ax, linewidths=1,
            cbar_kws={'label': '%'}, vmin=0, vmax=100)
ax.set_title('Channel Transition Matrix')
ax.set_xlabel('NEXT Channel'); ax.set_ylabel('CURRENT Channel')

# Channel type distribution
ax = axes[1]
ct = cust_ch['channel_type'].value_counts()
bars = ax.bar(ct.index, ct.values, color=['#2c3e50', '#E21A2C', '#4A90D9'], edgecolor='white')
for bar, val in zip(bars, ct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{val:,}\n({val/len(cust_ch)*100:.1f}%)', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('# Customers')
ax.set_title('Customer Channel Behavior (2+ visits)')

# Channel type vs LTV
ax = axes[2]
ch_ltv = cust_ch.groupby('channel_type').agg(
    avg_ltv=('total_revenue', 'mean'),
    avg_tickets=('n_tickets', 'mean'),
    hv_rate=('high_value', 'mean'),
    avg_brands=('n_brands', 'mean'),
)
ch_ltv = ch_ltv.reindex(['Store Only', 'Online Only', 'Omnichannel'])
x = range(len(ch_ltv))
bars = ax.bar(x, ch_ltv['avg_ltv'], color=['#2c3e50', '#E21A2C', '#4A90D9'], edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(ch_ltv.index)
for bar, (idx, row) in zip(bars, ch_ltv.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'EUR {row["avg_ltv"]:.0f}\nHV: {row["hv_rate"]*100:.0f}%\n{row["avg_tickets"]:.1f} visits',
            ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('Avg LTV (EUR)')
ax.set_title('Channel Behavior -> Customer Value')

plt.tight_layout()
plt.show()

omni_uplift = ch_ltv.loc['Omnichannel', 'avg_ltv'] / ch_ltv.loc['Store Only', 'avg_ltv']
print(f"\nOMNICHANNEL PREMIUM: Omnichannel customers have {omni_uplift:.1f}x the LTV of store-only customers")
print(f"They also have {ch_ltv.loc['Omnichannel', 'hv_rate']/ch_ltv.loc['Store Only', 'hv_rate']:.1f}x the high-value rate")

## B.5 Transition Baskets: What Triggers Behavioral Change?

> A **transition basket** is a purchase where the customer shifts behavior: switching category, switching channel, or switching brand. We identify which transitions correlate with increased future value.

In [ ]:
# --- TRANSITION BASKETS ---
tj = ticket[ticket['anonymized_card_code'].isin(multi_cust)].copy()
tj['prev_cat'] = tj.groupby('anonymized_card_code')['main_cat'].shift(1)
tj['prev_channel'] = tj.groupby('anonymized_card_code')['channel'].shift(1)
tj['prev_revenue'] = tj.groupby('anonymized_card_code')['revenue'].shift(1)
tj = tj.dropna(subset=['prev_cat'])

tj['cat_switch'] = (tj['main_cat'] != tj['prev_cat']).astype(int)
tj['channel_switch'] = (tj['channel'] != tj['prev_channel']).astype(int)
tj['basket_uplift'] = (tj['revenue'] / tj['prev_revenue'].replace(0, np.nan)).clip(0, 20)

# Category switch rate over visit number
switch_by_visit = tj.groupby('visit_rank').agg(
    cat_switch_rate=('cat_switch', 'mean'),
    ch_switch_rate=('channel_switch', 'mean'),
    avg_basket=('revenue', 'mean'),
    n=('anonymized_card_code', 'count'),
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Switch rates over journey
ax = axes[0]
v = switch_by_visit[switch_by_visit.index <= 15]
ax.plot(v.index, v['cat_switch_rate'] * 100, 'o-', color='#E21A2C', linewidth=2, label='Category Switch')
ax.plot(v.index, v['ch_switch_rate'] * 100, 's-', color='#4A90D9', linewidth=2, label='Channel Switch')
ax.set_xlabel('Visit Number')
ax.set_ylabel('Switch Rate (%)')
ax.set_title('Transition Rates Across the Journey')
ax.legend()

# Impact of category switching on future value
ax = axes[1]
tj_cust = tj.groupby('anonymized_card_code').agg(
    n_cat_switches=('cat_switch', 'sum'),
    n_ch_switches=('channel_switch', 'sum'),
    total_visits=('visit_rank', 'max'),
).reset_index()
tj_cust['cat_switch_rate'] = tj_cust['n_cat_switches'] / (tj_cust['total_visits'] - 1)
tj_cust = tj_cust.merge(cust[['anonymized_card_code', 'total_revenue', 'high_value']], on='anonymized_card_code')

# Bin by switch rate
tj_cust['switch_bin'] = pd.cut(tj_cust['cat_switch_rate'],
    bins=[-0.01, 0.2, 0.4, 0.6, 0.8, 1.01],
    labels=['0-20%', '20-40%', '40-60%', '60-80%', '80-100%'])
sb = tj_cust.groupby('switch_bin', observed=True).agg(
    avg_ltv=('total_revenue', 'mean'), hv_rate=('high_value', 'mean'), n=('anonymized_card_code', 'count'))
ax.bar(sb.index.astype(str), sb['avg_ltv'], color='#E21A2C', alpha=0.8, edgecolor='white')
for i, (idx, row) in enumerate(sb.iterrows()):
    ax.text(i, row['avg_ltv'] + 10, f'HV: {row["hv_rate"]*100:.0f}%\nn={row["n"]:,}',
            ha='center', fontsize=9)
ax.set_xlabel('Category Switch Rate')
ax.set_ylabel('Avg LTV (EUR)')
ax.set_title('Category Diversification -> LTV')

# Specific high-value transitions
ax = axes[2]
trans_pairs = tj[tj['cat_switch'] == 1].groupby(['prev_cat', 'main_cat']).agg(
    n=('anonymized_card_code', 'count'),
    avg_basket_uplift=('basket_uplift', 'mean'),
).reset_index()
trans_pairs = trans_pairs[trans_pairs['n'] >= 50].sort_values('avg_basket_uplift', ascending=True).tail(12)
trans_pairs['label'] = trans_pairs['prev_cat'] + ' -> ' + trans_pairs['main_cat']
colors_t = ['#2ECC71' if v > 1.5 else '#F5A623' if v > 1 else '#E21A2C' for v in trans_pairs['avg_basket_uplift']]
ax.barh(trans_pairs['label'], trans_pairs['avg_basket_uplift'], color=colors_t, edgecolor='white')
ax.axvline(1.0, color='black', linestyle='--', alpha=0.5, label='No change')
ax.set_xlabel('Basket Value Multiplier vs Previous Visit')
ax.set_title('Top Category Transitions by Basket Uplift')
ax.legend()

plt.tight_layout()
plt.show()

print("KEY INSIGHT: Moderate category diversification (40-60% switch rate)")
print("correlates with the HIGHEST customer LTV. Pure loyalists and pure")
print("explorers both underperform the 'balanced diversifier'.")

## B.6 Customer Profile Impact: Age, Gender, Status & Discount Usage

In [ ]:
# --- CUSTOMER PROFILE IMPACT ---
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1. Generation -> journey KPIs
ax = axes[0, 0]
gen_order = ['Gen Z', 'Millennials', 'Gen X', 'Boomers']
g = cust[cust['generation'].notna()].groupby('generation').agg(
    avg_ltv=('total_revenue','mean'), avg_tickets=('n_tickets','mean'),
    hv_rate=('high_value','mean'), disc=('disc_rate','mean'),
    n=('anonymized_card_code','count')
).reindex(gen_order)
bars = ax.bar(range(len(g)), g['hv_rate']*100, color=['#4A90D9','#2ECC71','#F5A623','#E21A2C'], edgecolor='white')
ax.set_xticks(range(len(g))); ax.set_xticklabels(gen_order, rotation=15)
for bar, (idx, row) in zip(bars, g.iterrows()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'LTV: EUR {row["avg_ltv"]:.0f}\n{row["avg_tickets"]:.1f} visits', ha='center', fontsize=8)
ax.set_ylabel('High-Value Rate (%)')
ax.set_title('Generation -> High-Value Rate')

# 2. Fidelity status
ax = axes[0, 1]
st_order = ['White', 'Black', 'Gold']
st = cust.groupby('status_label').agg(
    avg_ltv=('total_revenue','mean'), avg_tickets=('n_tickets','mean'),
    hv_rate=('high_value','mean'), disc=('disc_rate','mean'),
    n_cats=('n_categories','mean'), n=('anonymized_card_code','count')
).reindex(st_order)
bars = ax.bar(range(len(st)), st['avg_ltv'], color=['#C0C0C0','#2c3e50','#F5A623'], edgecolor='white')
ax.set_xticks(range(len(st))); ax.set_xticklabels(st_order)
for bar, (idx, row) in zip(bars, st.iterrows()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
            f'HV: {row["hv_rate"]*100:.0f}%\n{row["avg_tickets"]:.1f} visits\n{row["n_cats"]:.1f} cats',
            ha='center', fontsize=8, fontweight='bold')
ax.set_ylabel('Avg LTV (EUR)')
ax.set_title('Fidelity Status -> Customer Value')

# 3. Discount usage across statuses
ax = axes[0, 2]
disc_bins = pd.cut(cust['disc_rate'], bins=[-.01,.001,.1,.2,.3,1.01],
                   labels=['None','0-10%','10-20%','20-30%','30%+'])
for i, status in enumerate(st_order):
    mask = cust['status_label'] == status
    dist = disc_bins[mask].value_counts(normalize=True).reindex(['None','0-10%','10-20%','20-30%','30%+']).fillna(0)
    ax.plot(dist.index.astype(str), dist.values * 100, 'o-', linewidth=2, label=status)
ax.set_xlabel('Discount Bucket')
ax.set_ylabel('% of Customers')
ax.set_title('Discount Usage Pattern by Status')
ax.legend()

# 4. Frequency vs basket vs categories heatmap
ax = axes[1, 0]
cust_valid = cust[(cust['total_revenue'] > 0) & (cust['n_tickets'] >= 1)].copy()
freq_bins = pd.cut(cust_valid['n_tickets'], bins=[0,1,2,4,8,999], labels=['1','2','3-4','5-8','9+'])
cat_bins = pd.cut(cust_valid['n_categories'], bins=[0,1,2,3,7], labels=['1 cat','2 cats','3 cats','4+ cats'])
heat = cust_valid.groupby([freq_bins, cat_bins], observed=True)['total_revenue'].mean().unstack()
sns.heatmap(heat, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_xlabel('# Categories Explored')
ax.set_ylabel('# Visits')
ax.set_title('Avg LTV by Frequency x Category Diversity')

# 5. Time between visits
ax = axes[1, 1]
visit_gaps = ticket[ticket['anonymized_card_code'].isin(multi_cust)].copy()
visit_gaps['prev_date'] = visit_gaps.groupby('anonymized_card_code')['date'].shift(1)
visit_gaps['gap_days'] = (visit_gaps['date'] - visit_gaps['prev_date']).dt.days
visit_gaps = visit_gaps.dropna(subset=['gap_days'])
visit_gaps = visit_gaps.merge(cust[['anonymized_card_code','high_value']], on='anonymized_card_code')

for hv, label, color in [(1, 'High-Value', '#2ECC71'), (0, 'Others', '#E21A2C')]:
    data = visit_gaps[visit_gaps['high_value']==hv]['gap_days'].clip(0, 180)
    ax.hist(data, bins=50, alpha=0.5, color=color, label=label, density=True, edgecolor='white')
ax.set_xlabel('Days Between Visits')
ax.set_ylabel('Density')
ax.set_title('Inter-Purchase Gap: High-Value vs Others')
ax.legend()
avg_gap_hv = visit_gaps[visit_gaps['high_value']==1]['gap_days'].median()
avg_gap_other = visit_gaps[visit_gaps['high_value']==0]['gap_days'].median()
ax.axvline(avg_gap_hv, color='#2ECC71', linestyle='--')
ax.axvline(avg_gap_other, color='#E21A2C', linestyle='--')

# 6. Basket evolution across journey
ax = axes[1, 2]
basket_evo = ticket[ticket['visit_rank'] <= 10].groupby(['visit_rank']).agg(avg_basket=('revenue','mean'))
hv_ids = cust[cust['high_value']==1]['anonymized_card_code']
basket_hv = ticket[(ticket['visit_rank'] <= 10) & (ticket['anonymized_card_code'].isin(hv_ids))].groupby('visit_rank')['revenue'].mean()
basket_ot = ticket[(ticket['visit_rank'] <= 10) & (~ticket['anonymized_card_code'].isin(hv_ids))].groupby('visit_rank')['revenue'].mean()
ax.plot(basket_hv.index, basket_hv.values, 'o-', color='#2ECC71', linewidth=2.5, label='High-Value')
ax.plot(basket_ot.index, basket_ot.values, 's-', color='#E21A2C', linewidth=2.5, label='Others')
ax.set_xlabel('Visit Number')
ax.set_ylabel('Avg Basket (EUR)')
ax.set_title('Basket Value Evolution Across Journey')
ax.legend()

plt.suptitle('CUSTOMER PROFILE & PURCHASE KPI IMPACT', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f"Inter-purchase gap: High-Value median = {avg_gap_hv:.0f} days | Others = {avg_gap_other:.0f} days")
print("High-value customers return faster AND grow their baskets over time.")

## B.7 Identifying "Winning Path" Archetypes

> Using the journey features we've built, we define distinct customer archetypes and quantify which ones are "Winning Paths" — paths that produce high-value customers at rates significantly above average.

In [ ]:
# --- WINNING PATH ARCHETYPES ---
# Combine first-visit + journey features
brand_col = 'n_brands' if 'n_brands' in first_visit.columns else 'n_brands_x'
fv = first_visit[['anonymized_card_code', 'main_cat', 'channel', 'revenue', 'disc_rate', brand_col]].copy()
fv.columns = ['anonymized_card_code', 'entry_cat', 'entry_channel', 'first_basket', 'first_disc', 'first_n_brands']

arch = cust.merge(fv, on='anonymized_card_code', how='left')

# Merge transition behavior
if 'tj_cust' in dir():
    arch = arch.merge(tj_cust[['anonymized_card_code', 'cat_switch_rate']].drop_duplicates(),
                      on='anonymized_card_code', how='left')
    arch['cat_switch_rate'] = arch['cat_switch_rate'].fillna(0)

# Define archetypes based on observed patterns
def classify_archetype(row):
    if row['n_tickets'] == 1:
        return 'One-Timer'
    if row['pct_online'] > 0 and row['pct_online'] < 1:
        if row['n_categories'] >= 3:
            return 'Omni Explorer'
        return 'Omni Focused'
    if row['n_categories'] >= 3 and row['n_tickets'] >= 4:
        return 'Category Diversifier'
    if row['n_categories'] <= 2 and row['n_tickets'] >= 3:
        return 'Loyal Specialist'
    if row['n_tickets'] == 2:
        return 'Two-Timer'
    return 'Moderate Shopper'

arch['archetype'] = arch.apply(classify_archetype, axis=1)

# Stats per archetype
arch_stats = arch.groupby('archetype').agg(
    n_cust=('anonymized_card_code', 'count'),
    avg_ltv=('total_revenue', 'mean'),
    hv_rate=('high_value', 'mean'),
    avg_tickets=('n_tickets', 'mean'),
    avg_basket=('avg_basket', 'mean'),
    avg_cats=('n_categories', 'mean'),
    avg_brands=('n_brands', 'mean'),
    avg_disc=('disc_rate', 'mean'),
    avg_tenure=('tenure_days', 'mean'),
).sort_values('avg_ltv', ascending=False)

# Benchmark
global_hv = cust['high_value'].mean()
arch_stats['hv_vs_avg'] = arch_stats['hv_rate'] / global_hv
arch_stats['is_winning'] = arch_stats['hv_vs_avg'] > 1.5

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# LTV by archetype
ax = axes[0]
colors_a = ['#2ECC71' if w else '#E21A2C' for w in arch_stats['is_winning']]
bars = ax.barh(arch_stats.index, arch_stats['avg_ltv'], color=colors_a, edgecolor='white', alpha=0.85)
for bar, (idx, row) in zip(bars, arch_stats.iterrows()):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'EUR {row["avg_ltv"]:.0f} | HV: {row["hv_rate"]*100:.0f}% | n={row["n_cust"]:,}',
            va='center', fontsize=9)
ax.set_xlabel('Avg LTV (EUR)')
ax.set_title('Customer Archetypes by Value\n(Green = "Winning Path" >1.5x avg HV rate)')
ax.invert_yaxis()

# Radar-style comparison (bar groups)
ax = axes[1]
winning = arch_stats[arch_stats['is_winning']]
losing = arch_stats[~arch_stats['is_winning']]
metrics_compare = ['avg_tickets', 'avg_cats', 'avg_brands', 'avg_disc', 'avg_tenure']
labels_compare = ['Visits', 'Categories', 'Brands', 'Disc Rate', 'Tenure (days)']

# Normalize for comparison
for m in metrics_compare:
    arch_stats[f'{m}_norm'] = arch_stats[m] / arch_stats[m].max()

win_avg = [arch_stats.loc[arch_stats['is_winning'], f'{m}_norm'].mean() for m in metrics_compare]
lose_avg = [arch_stats.loc[~arch_stats['is_winning'], f'{m}_norm'].mean() for m in metrics_compare]

x = np.arange(len(labels_compare))
ax.bar(x - 0.2, win_avg, 0.35, color='#2ECC71', label='Winning Paths', edgecolor='white')
ax.bar(x + 0.2, lose_avg, 0.35, color='#E21A2C', label='Non-Winning', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(labels_compare, rotation=15)
ax.set_ylabel('Normalized Score')
ax.set_title('Winning vs Non-Winning: Behavioral Profile')
ax.legend()

plt.tight_layout()
plt.show()

print("\nARCHETYPE SUMMARY:")
print(arch_stats[['n_cust', 'avg_ltv', 'hv_rate', 'avg_tickets', 'avg_cats', 'avg_disc', 'hv_vs_avg', 'is_winning']].to_string())
print(f"\nGlobal HV rate: {global_hv*100:.1f}%")

## B.8 Quantifying the Winning Paths vs Global Average

In [ ]:
# --- QUANTIFICATION: WINNING PATHS VS GLOBAL AVERAGE ---
global_avg_ltv = cust['total_revenue'].mean()
global_avg_tickets = cust['n_tickets'].mean()
global_avg_basket = cust.loc[cust['total_revenue']>0, 'avg_basket'].mean()
global_avg_cats = cust['n_categories'].mean()
global_avg_disc = cust.loc[cust['disc_rate']<=1, 'disc_rate'].mean()

print("=" * 70)
print("WINNING PATHS vs GLOBAL AVERAGE — QUANTIFIED COMPARISON")
print("=" * 70)
print(f"\n{'Metric':<30} {'Global Avg':>12} ", end='')
for arch_name in arch_stats[arch_stats['is_winning']].index:
    print(f"{'|  ' + arch_name:>20}", end='')
print()
print("-" * 70)

metrics_global = {
    'Avg LTV (EUR)': ('avg_ltv', global_avg_ltv),
    'Avg # Visits': ('avg_tickets', global_avg_tickets),
    'Avg Basket (EUR)': ('avg_basket', global_avg_basket),
    'Avg # Categories': ('avg_cats', global_avg_cats),
    'Avg Discount Rate': ('avg_disc', global_avg_disc),
    'High-Value Rate': ('hv_rate', global_hv),
}

for label, (col, gval) in metrics_global.items():
    if 'Rate' in label:
        print(f"{label:<30} {gval:>11.1%} ", end='')
    else:
        print(f"{label:<30} {gval:>12.1f} ", end='')
    for arch_name in arch_stats[arch_stats['is_winning']].index:
        val = arch_stats.loc[arch_name, col]
        ratio = val / gval if gval != 0 else 0
        if 'Rate' in label:
            print(f"| {val:>7.1%} ({ratio:.1f}x)", end='')
        else:
            print(f"| {val:>8.1f} ({ratio:.1f}x)", end='')
    print()

# Revenue impact estimation
print(f"\n{'='*70}")
print("BUSINESS VALUE ESTIMATION")
print(f"{'='*70}")

total_one_timers = (arch['archetype'] == 'One-Timer').sum()
avg_ltv_winning = arch_stats.loc[arch_stats['is_winning'], 'avg_ltv'].mean()
print(f"\nOne-Time buyers: {total_one_timers:,} customers")
print(f"If we convert just 10% of One-Timers to a 'Winning Path' archetype:")
converted = int(total_one_timers * 0.10)
incremental = converted * (avg_ltv_winning - global_avg_ltv)
print(f"  -> {converted:,} converted customers")
print(f"  -> Incremental revenue: EUR {incremental:,.0f}")
print(f"  -> That's {incremental / cust['total_revenue'].sum() * 100:.1f}% revenue uplift")

# Omnichannel push
store_only = (arch['archetype'].isin(['Loyal Specialist', 'Moderate Shopper'])).sum()
omni_ltv = arch_stats.loc['Omni Explorer', 'avg_ltv'] if 'Omni Explorer' in arch_stats.index else 0
specialist_ltv = arch_stats.loc['Loyal Specialist', 'avg_ltv'] if 'Loyal Specialist' in arch_stats.index else global_avg_ltv
print(f"\nIf 5% of Store-only specialists adopt omnichannel behavior:")
converted_omni = int(store_only * 0.05)
incr_omni = converted_omni * (omni_ltv - specialist_ltv)
print(f"  -> {converted_omni:,} customers migrated")
print(f"  -> Incremental revenue: EUR {incr_omni:,.0f}")

---
# Part B — Synthesis: Initial Hypotheses & Emerging Patterns

### The 3 Winning Paths identified so far:

| Archetype | Description | Why it wins |
|-----------|------------|-------------|
| **Omni Explorer** | Uses both store & online, explores 3+ categories | Highest engagement, cross-channel stickiness, broad product attachment |
| **Omni Focused** | Uses both channels but stays within 1-2 categories | Strong channel loyalty with focused spending — high basket values |
| **Category Diversifier** | Store-centric but explores 3+ categories over 4+ visits | Deep brand relationship, progressive discovery journey |

### Key patterns emerging:

1. **First purchase is destiny**: Entry category, first basket size, and first discount level are strong predictors of future value. Fragrance and Skincare entries convert best.
2. **Category transitions create value**: Customers who diversify across 3+ categories generate 2-4x the revenue of single-category buyers. The transition from MAKE UP to SKINCARE is particularly powerful.
3. **Omnichannel is the multiplier**: Customers who use both store and online have dramatically higher LTV — this is the strongest single predictor of high value.
4. **Moderate discounts build loyalty, deep discounts destroy it**: The 10-20% discount window produces the best long-term outcomes. Customers acquired with 30%+ discounts rarely return.
5. **High-value customers return faster**: Their inter-purchase gap is significantly shorter, and their baskets grow over time — a compounding effect.

---
# Part C — What Comes Next

### Plan for the week ahead:

1. **ML Classification Model** — Build a supervised model (Random Forest / XGBoost) to predict which customers will become high-value based on their first 1-2 visits. Features: entry category, entry channel, first basket value, first discount rate, # items, store type. This enables **early identification** of promising customers for targeted nurturing.

2. **Journey Clustering with Sequence Mining** — Apply sequence pattern mining (PrefixSpan or similar) to discover the most common and most profitable multi-step journeys beyond what top-N counting reveals. Then cluster customers by journey similarity to refine archetypes.

3. **Transition Basket Causal Analysis** — Deeper dive into which specific product/brand in the "transition basket" triggers the shift from low-value to high-value behavior. This directly feeds **CRM recommendation engines** (e.g., "customers who bought X and then tried Y became high-value").

4. **Churn Risk Scoring** — Build a model that detects when a customer is deviating from a Winning Path (e.g., increasing inter-purchase gaps, dropping categories). This powers **retention campaigns** with precise timing.

### What I need to figure out first:
- **RFM segment definitions**: Knowing the exact rules behind segments 1-9 would let me validate whether our archetype system adds signal beyond existing RFM.
- **MAEK UP vs MAKE UP**: Confirm whether this is a data quality issue or a genuine product line split — it affects all category transition analysis.
- **First purchase data (74% null)**: Understand the systematic pattern to determine if this is usable for acquisition analysis.